In [1]:
pip install boto3

   ---------------------------------------- 0.0/139.3 kB ? eta -:--:--
   -- ------------------------------------- 10.2/139.3 kB ? eta -:--:--
   -------- ------------------------------ 30.7/139.3 kB 660.6 kB/s eta 0:00:01
   -------------------------- ------------- 92.2/139.3 kB 1.1 MB/s eta 0:00:01
   --------------------------------- ---- 122.9/139.3 kB 722.1 kB/s eta 0:00:01
   -------------------------------------- 139.3/139.3 kB 690.3 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import boto3
from dotenv import load_dotenv
import os
from botocore.config import Config

# ✅ Load environment variables from .env file
load_dotenv()

# ✅ Read values from environment
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_session_token = os.getenv("AWS_SESSION_TOKEN")  # optional
aws_region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

# ✅ Create default boto3 session using loaded credentials
boto3.setup_default_session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    region_name=aws_region
)

print("✅ Boto3 default session configured successfully using .env file.")

# ✅ Test: List Kinesis streams to verify credentials
kinesis_client = boto3.client('kinesis')
response = kinesis_client.list_streams()
print("Available Streams:", response.get('StreamNames'))

Available Streams: []


#### 1 — Create S3 bucket (GUI + CLI)

GUI

- Console → S3 → Create bucket.

- Name: data-forge-team4-yourname, region: us-east-1. Create.

- In bucket, Create folders: raw/, processed/, scripts/, checkpoint/, query-results/.

- Upload /mnt/data/mobile-logs.csv into raw/mobile-logs.csv.

In [2]:
import boto3
from botocore.exceptions import ClientError

# Initialize the S3 client
s3_client = boto3.client('s3', region_name='us-east-1')

# Bucket name
bucket_name = 'data-forge-bkt1'

try:
    # 1. Create the S3 bucket (no LocationConstraint for us-east-1)
    s3_client.create_bucket(Bucket=bucket_name)
    print(f"Bucket '{bucket_name}' created.")
except ClientError as e:
    print(f"Error creating bucket: {e}")

# 2. Create "folders" by uploading empty objects with trailing slashes
folders = ['raw/', 'processed/', 'scripts/', 'checkpoint/', 'query-results/']

for folder in folders:
    try:
        s3_client.put_object(Bucket=bucket_name, Key=folder)
        print(f"Folder '{folder}' created.")
    except ClientError as e:
        print(f"Error creating folder '{folder}': {e}")

# 3. Upload the file
file_path = 'mobile-logs.csv'
object_key = 'raw/mobile-logs.csv'

try:
    s3_client.upload_file(file_path, bucket_name, object_key)
    print(f"File '{file_path}' uploaded to s3://{bucket_name}/{object_key}")
except ClientError as e:
    print(f"Error uploading file: {e}")


Bucket 'data-forge-bkt1' created.
Folder 'raw/' created.
Folder 'processed/' created.
Folder 'scripts/' created.
Folder 'checkpoint/' created.
Folder 'query-results/' created.
File 'mobile-logs.csv' uploaded to s3://data-forge-bkt1/raw/mobile-logs.csv


#### Checking if file exists at path

In [2]:
import os

file_name = 'mobile-logs.csv' # Replace with your file's name or path

if os.path.isfile(file_name):
    print(f"The file '{file_name}' exists.")
else:
    print(f"The file '{file_name}' does not exist.")

The file 'mobile-logs.csv' exists.


#### 2 — Create Kinesis Data Stream (GUI + CLI)

GUI

- Console → Kinesis → Data Streams → Create data stream.

- Name: mobile-log-stream, shards: 1. Create and wait until ACTIVE.

In [5]:
import boto3
import time
from botocore.exceptions import ClientError

# Initialize the Kinesis client
kinesis_client = boto3.client('kinesis', region_name='us-east-1')

stream_name = 'kinesis-mobile-log-stream'
shard_count = 1

# 1. Create Kinesis stream
try:
    response = kinesis_client.create_stream(
        StreamName=stream_name,
        ShardCount=shard_count
    )
    print(f"Creating Kinesis stream '{stream_name}' with {shard_count} shard(s)...")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceInUseException':
        print(f"Stream '{stream_name}' already exists.")
    else:
        print(f"Error creating stream: {e}")
        exit(1)

# 2. Wait until the stream becomes ACTIVE
max_retries = 10
wait_time = 5  # seconds

for attempt in range(max_retries):
    try:
        response = kinesis_client.describe_stream(StreamName=stream_name)
        status = response['StreamDescription']['StreamStatus']
        print(f"Attempt {attempt + 1}: Stream status is '{status}'")
        if status == 'ACTIVE':
            print(f"Stream '{stream_name}' is ACTIVE and ready to use.")
            break
        time.sleep(wait_time)
    except ClientError as e:
        print(f"Error describing stream: {e}")
        time.sleep(wait_time)
else:
    print(f"Stream '{stream_name}' did not become ACTIVE after {max_retries} attempts.")


Creating Kinesis stream 'kinesis-mobile-log-stream' with 1 shard(s)...
Attempt 1: Stream status is 'CREATING'
Attempt 2: Stream status is 'ACTIVE'
Stream 'kinesis-mobile-log-stream' is ACTIVE and ready to use.


#### 3 — Create IAM role for Glue & other services (GUI + CLI)

The Glue role must trust Glue and allow read Kinesis, write S3, and CloudWatch Logs.

GUI

- Console → IAM → Roles → Create role.

- Choose Glue as trusted service.

- Attach managed policies: AmazonS3FullAccess (or narrower S3 policy), AmazonKinesisReadOnlyAccess, CloudWatchLogsFullAccess.

- Name role glue-Stream-Execution-Role. Create.

In [16]:
import json
import boto3
from botocore.exceptions import ClientError

# Initialize IAM client
iam_client = boto3.client('iam')

# Role name
role_name = 'glue-Stream-Execution-Role'

# Assume role trust policy for Glue
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "glue.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

# Managed policies to attach
managed_policies = [
    'arn:aws:iam::aws:policy/AmazonS3FullAccess',
    'arn:aws:iam::aws:policy/AmazonKinesisReadOnlyAccess',
    'arn:aws:iam::aws:policy/CloudWatchLogsFullAccess',
    'arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole'
]

# 1. Create the IAM Role
try:
    response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='IAM role for AWS Glue to read Kinesis, write to S3, and log to CloudWatch'
    )
    print(f"Role '{role_name}' created successfully.")
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        print(f"Role '{role_name}' already exists.")
    else:
        print(f"Error creating role: {e}")
        exit(1)

# 2. Attach managed policies
for policy_arn in managed_policies:
    try:
        iam_client.attach_role_policy(
            RoleName=role_name,
            PolicyArn=policy_arn
        )
        print(f"Attached policy: {policy_arn}")
    except ClientError as e:
        print(f"Error attaching policy {policy_arn}: {e}")


Role 'glue-Stream-Execution-Role' created successfully.
Attached policy: arn:aws:iam::aws:policy/AmazonS3FullAccess
Attached policy: arn:aws:iam::aws:policy/AmazonKinesisReadOnlyAccess
Attached policy: arn:aws:iam::aws:policy/CloudWatchLogsFullAccess
Attached policy: arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole


#### 4 — Glue Streaming Job (Spark) — create script, upload, create job (GUI + CLI)
- 4.1 Prepare Glue script (local)

    - Create glue_stream_mobile_logs.py — this reads JSON text from Kinesis, parses it, adds ingestion_time and partitions, writes Parquet to S3. Put this file into your S3 scripts folder.

- 4.2 Create Glue job (GUI)

    - Console → AWS Glue → Jobs → Create job.

    - Name: Glue_Stream_MobileLogs.

    - Choose the role: glue-Stream-Execution-Role.

    - Glue version: pick one with Spark streaming support (e.g., Glue 3.0/4.0).

    - Type: Spark (enable streaming/continuous). Some consoles label it "Job type: Spark (Streaming)".

    - For Script file: choose the S3 location where you uploaded glue_stream_mobile_logs.py.

    - Worker type and number: start small (G.1X, 1-2 workers).

    - Save and Run. The job will connect to the Kinesis stream and wait for data.

In [8]:
import boto3
import json
from botocore.exceptions import ClientError

# ---- Configuration ----
bucket_name = 'data-forge-bkt1'
script_file = 'glue_stream_mobile_logs.py'
script_local_path = f'./{script_file}'  # Assumes script is in the current directory
script_s3_key = f'scripts/{script_file}'
script_s3_path = f's3://{bucket_name}/{script_s3_key}'
job_name = 'Glue_Stream_MobileLogs'
role_name = 'glue-Stream-Execution-Role'
glue_version = '3.0'
python_version = '3'
max_capacity = 2  # Adjust as needed
region = 'us-east-1'

# ---- Upload Script to S3 ----
s3 = boto3.client('s3', region_name=region)

try:
    s3.upload_file(script_local_path, bucket_name, script_s3_key)
    print(f"✅ Script uploaded to S3: {script_s3_path}")
except ClientError as e:
    print(f"❌ Failed to upload script to S3: {e}")
    exit(1)

# ---- Create Glue Job ----
glue = boto3.client('glue', region_name=region)

try:
    response = glue.create_job(
        Name=job_name,
        Role=role_name,
        ExecutionProperty={
            'MaxConcurrentRuns': 1
        },
        Command={
            'Name': 'gluestreaming',
            'ScriptLocation': script_s3_path,
            'PythonVersion': python_version
        },
        GlueVersion=glue_version,
        MaxCapacity=max_capacity,
        Description='Streaming job to read mobile logs from Kinesis and write to S3 in Parquet format.',
         DefaultArguments={
            '--TempDir': f's3://{bucket_name}/temp/',
            '--STREAM_ARN': 'arn:aws:kinesis:us-east-1:096212910226:stream/kinesis-mobile-log-stream',
            '--OUTPUT_PATH': f's3://{bucket_name}/processed',
            '--job-language': 'python',
            '--enable-continuous-cloudwatch-log': 'true',
            '--enable-metrics': 'true'
        },
        Tags={
            'Project': 'DataForge',
            'Team': 'Team4'
        }
    )
    print(f"✅ Glue job '{job_name}' created successfully.")
except ClientError as e:
    if e.response['Error']['Code'] == 'AlreadyExistsException':
        print(f"⚠️ Glue job '{job_name}' already exists.")
    else:
        print(f"❌ Failed to create Glue job: {e}")
        exit(1)

# ---- Start Glue Job Run ----
try:
    start_response = glue.start_job_run(JobName=job_name)
    run_id = start_response['JobRunId']
    print(f"🚀 Glue job '{job_name}' started successfully. Run ID: {run_id}")
except ClientError as e:
    print(f"❌ Failed to start Glue job: {e}")


✅ Script uploaded to S3: s3://data-forge-bkt1/scripts/glue_stream_mobile_logs.py
✅ Glue job 'Glue_Stream_MobileLogs' created successfully.
🚀 Glue job 'Glue_Stream_MobileLogs' started successfully. Run ID: jr_8d8522f2d9d10ff5f332666ac487035ebd60c1a03c4e9f6c9ffc754c3b23d539


#### 5 — Producer: simulate streaming from your CSV (local script)

- Send CSV rows as JSON to Kinesis.

In [14]:
import csv, json, time, boto3

STREAM_NAME = "kinesis-mobile-log-stream"
CSV_PATH = "mobile-logs.csv"  # user's dataset path
SLEEP = 0.05

kinesis = boto3.client("kinesis", region_name="us-east-1")

with open(CSV_PATH, 'r') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        data = json.dumps(row)
        kinesis.put_record(StreamName=STREAM_NAME, Data=data.encode('utf-8'), PartitionKey=str(i%10))
        if i % 50 == 0:
            print("sent", i)
        time.sleep(SLEEP)



sent 0
sent 50
sent 100
sent 150
sent 200
sent 250
sent 300
sent 350
sent 400
sent 450
sent 500
sent 550
sent 600
sent 650
sent 700
sent 750
sent 800
sent 850
sent 900
sent 950


### Glue updated

In [13]:
import boto3
from botocore.exceptions import ClientError

# Initialize clients
s3 = boto3.client('s3')
kinesis = boto3.client('kinesis')
glue = boto3.client('glue')

bucket_name = 'data-forge-bkt1'
role_name = 'glue-Stream-Execution-Role'
job_name = 'Glue_Stream_MobileLogs'
script_file = 'glue_stream_mobile_logs.py'
script_local_path = f'./{script_file}'  # Assumes script is in the current directory
script_s3_key = f'scripts/{script_file}'
script_s3_path = f's3://{bucket_name}/scripts/{script_file}'
python_version = '3'
glue_version = '4.0'
max_capacity = 1.0
region = 'us-east-1'


# ---- Upload Script to S3 ----
s3 = boto3.client('s3', region_name=region)

try:
    s3.upload_file(script_local_path, bucket_name, script_s3_key)
    print(f"✅ Script uploaded to S3: {script_s3_path}")
except ClientError as e:
    print(f"❌ Failed to upload script to S3: {e}")
    exit(1)

# 1️⃣ Ensure /temp/ folder exists in S3
try:
    s3.put_object(Bucket=bucket_name, Key='temp/')
    print(f"✅ Temp directory ensured in S3: s3://{bucket_name}/temp/")
except ClientError as e:
    print(f"❌ Failed to create /temp/ folder: {e}")
    exit(1)

# 2️⃣ Fetch the Kinesis Stream ARN dynamically
try:
    stream_name = "kinesis-mobile-log-stream"
    response = kinesis.describe_stream(StreamName=stream_name)
    stream_arn = response['StreamDescription']['StreamARN']
    print(f"✅ Kinesis Stream ARN fetched: {stream_arn}")
except ClientError as e:
    print(f"❌ Failed to get Kinesis stream ARN: {e}")
    exit(1)

# 3️⃣ Create or update the Glue job
try:
    response = glue.create_job(
        Name=job_name,
        Role=role_name,
        ExecutionProperty={'MaxConcurrentRuns': 1},
        Command={
            'Name': 'gluestreaming',
            'ScriptLocation': script_s3_path,
            'PythonVersion': python_version
        },
        GlueVersion=glue_version,
        MaxCapacity=max_capacity,
        Description='Reusable streaming job to read from Kinesis and write to S3 in CSV format.',
        DefaultArguments={
            '--TempDir': f's3://{bucket_name}/temp/',
            '--STREAM_ARN': stream_arn,  # ✅ dynamically fetched
            '--OUTPUT_PATH': f's3://{bucket_name}/processed/',
            '--job-language': 'python',
            '--enable-continuous-cloudwatch-log': 'true',
            '--enable-metrics': 'true'
        },
        Tags={'Project': 'DataForge', 'Team': 'Team4'}
    )
    print(f"✅ Glue job '{job_name}' created successfully.")
except ClientError as e:
    if e.response['Error']['Code'] == 'AlreadyExistsException':
        print(f"⚠️ Glue job '{job_name}' already exists.")
    else:
        print(f"❌ Failed to create Glue job: {e}")
        exit(1)


✅ Script uploaded to S3: s3://data-forge-bkt1/scripts/glue_stream_mobile_logs.py
✅ Temp directory ensured in S3: s3://data-forge-bkt1/temp/
✅ Kinesis Stream ARN fetched: arn:aws:kinesis:us-east-1:801341413974:stream/kinesis-mobile-log-stream
✅ Glue job 'Glue_Stream_MobileLogs' created successfully.


In [12]:
# ---- Start Glue Job Run ----
try:
    start_response = glue.start_job_run(JobName=job_name)
    run_id = start_response['JobRunId']
    print(f"🚀 Glue job '{job_name}' started successfully. Run ID: {run_id}")
except ClientError as e:
    print(f"❌ Failed to start Glue job: {e}")

🚀 Glue job 'Glue_Stream_MobileLogs' started successfully. Run ID: jr_1efbf742e8e15b8ebb31b2e231e9e7ef92146fff4b1282b16707e2c39a8749a9


#### 6 — Create Glue Crawler & Data Catalog (GUI + CLI)

Goal: Automatically infer schema and register processed data for Athena queries.

GUI:

Go to AWS Glue → Crawlers → Add crawler.

Name: crawler-mobile-processed

Source: S3 path → s3://data-forge-team4-yourname/processed/

IAM Role: glue-Stream-Execution-Role

Run frequency: On Demand

Target database: mobile_logs_db (create if missing)

Run crawler → after completion you’ll see a table such as processed_mobile_logs.

In [18]:
import boto3
from botocore.exceptions import ClientError

# Initialize Glue client
glue = boto3.client('glue')

# Define parameters

bucket_name = 'data-forge-bkt1'
database_name = "mobile_logs_db"
crawler_name = "crawler-mobile-processed"
role_name = "glue-Stream-Execution-Role"
s3_path = f"s3://{bucket_name}/processed/"

# Step 1: Create Database
try:
    print(f"Creating database '{database_name}'...")
    response = glue.create_database(
        DatabaseInput={
            'Name': database_name,
            'Description': 'Database for processed mobile logs'
        }
    )
    print("✅ Database created successfully.")
except ClientError as e:
    if e.response['Error']['Code'] == 'AlreadyExistsException':
        print(f"⚠️ Database '{database_name}' already exists.")
    else:
        print(f"❌ Failed to create database: {e}")
        raise

# Step 2: Create Crawler
try:
    print(f"Creating crawler '{crawler_name}'...")
    response = glue.create_crawler(
        Name=crawler_name,
        Role=role_name,
        DatabaseName=database_name,
        Description='Crawler for processed mobile logs data',
        Targets={
            'S3Targets': [
                {'Path': s3_path}
            ]
        },
        TablePrefix='df_',
        RecrawlPolicy={
            'RecrawlBehavior': 'CRAWL_NEW_FOLDERS_ONLY'
        },
         SchemaChangePolicy={
            'UpdateBehavior': 'LOG',
            'DeleteBehavior': 'LOG'
        }
    )
    print("✅ Crawler created successfully.")
except ClientError as e:
    if e.response['Error']['Code'] == 'AlreadyExistsException':
        print(f"⚠️ Crawler '{crawler_name}' already exists.")
    else:
        print(f"❌ Failed to create crawler: {e}")
        raise

# Step 3: Start Crawler
try:
    print(f"Starting crawler '{crawler_name}'...")
    response = glue.start_crawler(Name=crawler_name)
    print("🚀 Crawler started successfully.")
except ClientError as e:
    if e.response['Error']['Code'] == 'CrawlerRunningException':
        print(f"⚠️ Crawler '{crawler_name}' is already running.")
    else:
        print(f"❌ Failed to start crawler: {e}")
        raise


Creating database 'mobile_logs_db'...
✅ Database created successfully.
Creating crawler 'crawler-mobile-processed'...
✅ Crawler created successfully.
Starting crawler 'crawler-mobile-processed'...
🚀 Crawler started successfully.
